# 04 — Évaluation à grande échelle des modèles

## Objectif du notebook

Ce notebook applique le protocole expérimental défini et validé dans les étapes précédentes à un échantillon de grande taille issu du jeu de données MediQAI.

Un échantillon reproductible de **5 000 questions MCQU** est sélectionné à partir du split `train`. Le même ensemble de questions et le même `prompt_v3` sont utilisés pour évaluer les deux modèles étudiés :

- **Gemini** ;
- **OpenAI**.

L'objectif est de comparer leurs performances dans des conditions expérimentales identiques et d'identifier les réponses susceptibles de faire l'objet d'une analyse qualitative des hallucinations.

L'évaluation porte notamment sur :

- l'exactitude des réponses ;
- le score F1 macro ;
- les erreurs de réponse ;
- les désaccords entre les modèles ;
- la répartition des erreurs selon les spécialités médicales ;
- les cas dans lesquels les deux modèles échouent sur une même question.

Les réponses incorrectes sont finalement exportées afin de constituer le corpus utilisé pour l'analyse détaillée des hallucinations.

## 1. Initialisation de l'environnement

Les bibliothèques nécessaires à la manipulation des données, aux appels API et à l'évaluation des modèles sont importées.

La racine du projet est déterminée dynamiquement afin de permettre l'exécution du notebook depuis différents répertoires. Elle est ensuite ajoutée au chemin Python pour rendre accessibles les fonctions développées dans le dossier `src`.

Plusieurs fonctions définies lors des étapes précédentes sont ainsi réutilisées pour :

- préparer les questions et construire les prompts ;
- interroger les différents LLM ;
- exécuter les expériences par lots ;
- comparer les réponses des modèles ;
- extraire et sauvegarder les erreurs.

Cette organisation permet de séparer la logique expérimentale du notebook des fonctions réutilisables du projet.

In [1]:
from pathlib import Path
import os
import re
import time

import pandas as pd
from dotenv import load_dotenv

In [2]:
from pathlib import Path
import sys

if Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [3]:
# importation des fonctions crées à partir du notebook 03_evaluation_finale stockées dans le dossier src
from src.prompt import prepare_sample
from src.llm_inference import call_llm, run_single_experiment
from src.evaluation import build_model_comparison, get_model_errors, save_csv, save_parquet
from src.experiment_runner import run_experiment_batch

## 2. Définition des données et des répertoires de résultats

Les données préparées dans les notebooks précédents sont chargées depuis le répertoire `data/processed`.

Les résultats de cette expérimentation sont enregistrés séparément dans le répertoire `final_evaluation` afin de conserver les réponses brutes et les fichiers nécessaires aux analyses ultérieures.

In [4]:

PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "final_evaluation"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TRAIN_DATA_PATH = (
    PROCESSED_DIR
    / "benchmark_mcqu_train.parquet"
)

## 3. Construction de l'échantillon d'évaluation

Le dataset MCQU préparé précédemment est chargé puis un échantillon de **5 000 questions** est sélectionné aléatoirement.

Une graine fixe (`random_state = 42`) est utilisée afin de rendre l'échantillonnage reproductible. Les deux modèles seront évalués sur exactement le même ensemble de questions, ce qui permet une comparaison directe de leurs performances.

Cette expérimentation représente une augmentation importante de l'échelle par rapport au pilote et au pré-benchmark réalisés dans le notebook précédent.

In [5]:
df_trn = pd.read_parquet(
    TRAIN_DATA_PATH
)

In [6]:
RANDOM_SEED = 42
PROMPT_SAMPLE_SIZE = 5002
df_sample = (
    df_trn.sample(
        n=PROMPT_SAMPLE_SIZE,
        random_state=RANDOM_SEED
    )
)

## 4. Configuration des modèles évalués

Les clients API Gemini et OpenAI sont initialisés à partir des clés stockées dans le fichier `.env`.

Les modèles utilisés sont :

- **Gemini :** `gemini-3.6-flash`
- **OpenAI :** `gpt-5.4-mini-2026-03-17`

Les deux modèles sont interrogés indépendamment mais selon le même protocole expérimental afin de limiter les différences liées aux conditions d'évaluation.

In [7]:
# Chargement de la clé API du fichier .env
from google import genai
from openai import OpenAI
env_path = Path.cwd()/ ".env"
load_dotenv(env_path)

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Initialisation des clients Gemni et OpenAI
gemini_client = genai.Client(
    api_key=GEMINI_API_KEY
)
llm_name_1 = "gemini"
model_name_1="gemini-3.6-flash"

openai_client = OpenAI(
    api_key=OPENAI_API_KEY
)
llm_name_2 = "openai"
model_name_2="gpt-5.4-mini-2026-03-17"

## 5. Préparation des expériences

La fonction `prepare_sample` transforme l'échantillon sélectionné en un tableau directement exploitable par le pipeline d'inférence.

Elle construit notamment le prompt à partir du `question_context` et associe à chaque expérience les informations nécessaires à son évaluation, telles que l'identifiant de la question et la réponse de référence.

Le **`prompt_v3` sélectionné lors de l'étape précédente** est utilisé pour l'ensemble de cette expérimentation. Le format demandé comprend une réponse, une justification médicale et un niveau de confiance déclaré.

In [8]:
df_prepared_sample = prepare_sample(df_sample)

In [9]:
display(df_sample)

,sample_id,id,configuration,split,clinical_case,question,answer_a,answer_b,answer_c,answer_d,answer_e,choices,choices_text,reference_letter,reference_answer,medical_subject,question_type,task,question_context
9571,mcqu_train_13934,13934,mcqu,train,,"Dans un test de dépistage d'une maladie, la pr...",La spécifité,La sensibilité,La prévalence,L'incidence,La valeur prédictive positive,"{'A': 'La spécifité', 'B': 'La sensibilité', '...",A. La spécifité\nB. La sensibilité\nC. La prév...,B,La sensibilité,Epidemiology,Understanding,QCU,Question :\nDans un test de dépistage d'une ma...
9502,mcqu_train_18036,18036,mcqu,train,"Une femme de 47 ans, nulligeste, jusque là nor...","Parmi les examens complémentaires suivants, qu...",Frottis cervico-vaginaux de cytodétection,Biopsie d'endomètre,Radiographie de l'abdomen sans préparation,Numération - formule sanguine et taux d'hémogl...,Hystérosalpingographie,{'A': 'Frottis cervico-vaginaux de cytodétecti...,A. Frottis cervico-vaginaux de cytodétection\n...,D,Numération - formule sanguine et taux d'hémogl...,Gynecology and Obstetrics,Reasoning,QCU,"Cas clinique :\nUne femme de 47 ans, nulligest..."
3379,mcqu_train_9861,9861,mcqu,train,,Lors de la prescription d'un collyre mydriatiq...,Prendre le tonus oculaire,Vérifier l'état de l'angle camérulaire,Vérifier l'état de la rétine périphérique,Vérifier l'état de la papille,Vérifier l'état de la macula,"{'A': 'Prendre le tonus oculaire', 'B': 'Vérif...",A. Prendre le tonus oculaire\nB. Vérifier l'ét...,B,Vérifier l'état de l'angle camérulaire,Ophthalmology,Understanding,QCU,Question :\nLors de la prescription d'un colly...
5407,mcqu_train_1065,1065,mcqu,train,,Parmi les affirmations suivantes la(lesquelles...,1+2+3,1+3,2+4,4,1+2+3+4,"{'A': '1+2+3', 'B': '1+3', 'C': '2+4', 'D': '4...",A. 1+2+3\nB. 1+3\nC. 2+4\nD. 4\nE. 1+2+3+4,E,1+2+3+4,Cardiology,Understanding,QCU,Question :\nParmi les affirmations suivantes l...
7815,mcqu_train_27194,27194,mcqu,train,"Patiente âgée de 62 ans, sans antécédents path...",Le diagnostic le plus probable est : (Cocher l...,Hypothyroïdie primaire post ménopausique,Hypothyroïdie primaire par thyroïdite de Hashi...,Hypothyroïdie primaire par thyroïdite subaigüe,Hypothyroïdie primaire par thyroïdite de Riedel,Hypothyroïdie primaire par thyroïdite atrophique,{'A': 'Hypothyroïdie primaire post ménopausiqu...,A. Hypothyroïdie primaire post ménopausique\nB...,B,Hypothyroïdie primaire par thyroïdite de Hashi...,Endocrinology and Metabolism,Reasoning,QCU,"Cas clinique :\nPatiente âgée de 62 ans, sans ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10004,mcqu_train_2742,2742,mcqu,train,,Parmi les affirmations suivantes concernant la...,1+2+3,1+3,2+4,4,1+2+3+4,"{'A': '1+2+3', 'B': '1+3', 'C': '2+4', 'D': '4...",A. 1+2+3\nB. 1+3\nC. 2+4\nD. 4\nE. 1+2+3+4,A,1+2+3,Hematology,Understanding,QCU,Question :\nParmi les affirmations suivantes c...
3049,mcqu_train_21599,21599,mcqu,train,,(cochez la réponse fausse) La présence d'une t...,Un cancer profond,Trouble congénital de l’hémostase Absences des...,Une hémopathie maligne,La prise d'oestro-progestatifs,Un ulcère gastrique,"{'A': 'Un cancer profond', 'B': 'Trouble congé...",A. Un cancer profond\nB. Trouble congénital de...,E,Un ulcère gastrique,Cardiology,Understanding,QCU,Question :\n(cochez la réponse fausse) La prés...
9631,mcqu_train_12513,12513,mcqu,train,,Les résultats d'une mesure des gaz du sang art...,Acidose métabolique compensée,Alcalose ventilatoire non compensée,Acidose ventilatoire compensée,Alcalose ventilatoire compensée,Alcalose métabolique non compensée,"{'A': 'Acidose métabolique compensée', 'B': 'A...",A. Acidose métabolique compensée\nB. Alcalose ...,C,Acidose ventilatoire compensée,Pulmonology,Reasoning,QCU,Question :\nLes résultats d'une mesure des gaz...
1273,mcqu_train_24867,24867,mcqu,train,,(cochez la réponse fausse) La diarrhée par tox...,Salmonella mineure.,Staphylococcus auréus.,Streptococcus pyogènes.,Clostridium perfring

## 6. Exécution du benchmark avec Gemini

Les 5 000 questions préparées sont soumises au modèle Gemini.

Les résultats sont enregistrés progressivement au format Parquet. Cette sauvegarde permet de conserver les réponses déjà obtenues en cas d'interruption et d'éviter de répéter inutilement des appels API.

Chaque résultat contient notamment la réponse brute du modèle, la lettre prédite, la justification, la confiance déclarée, la validité du format, la latence et l'éventuelle erreur technique.

In [10]:
GEMINI_BENCHMARK_PATH = (
    RESULTS_DIR / "gemini_prompt_v3_benchmark.parquet"
)

#df_gemini_benchmark = run_experiment_batch(
#    experiments=df_prepared_sample,
#    llm_name=llm_name_1,
#    runner=run_single_experiment,
#    model_name=model_name_1,
#    llm_client=gemini_client,
#    output_path=GEMINI_BENCHMARK_PATH,
#    pause_seconds=1,
#)

## 7. Exécution du benchmark avec OpenAI

Le même ensemble de 5 000 questions est ensuite soumis au modèle OpenAI en utilisant le même `prompt_v3`.

La conservation du même échantillon et du même protocole permet de comparer directement les prédictions des deux modèles question par question.

In [11]:
OPENAI_BENCHMARK_PATH = (
    RESULTS_DIR / "openai_prompt_v3_benchmark.parquet"
)

df_openai_benchmark = run_experiment_batch(
    experiments=df_prepared_sample,
    llm_name=llm_name_2,
    runner=run_single_experiment,
    model_name=model_name_2,
    llm_client=openai_client,
    output_path=OPENAI_BENCHMARK_PATH,
    pause_seconds=1,
)

Déjà terminées : 5001
À exécuter : 1
[1/1] mcqu_train_999
Prédiction : C | Référence : C | Correcte : True | Erreur : None


## 8. Chargement et contrôle des résultats

Les résultats générés par les deux modèles sont rechargés depuis les fichiers Parquet afin de réaliser les analyses sans répéter les appels API.

Avant le calcul des métriques, les résultats sont contrôlés afin d'identifier les expériences pour lesquelles aucune lettre de réponse n'a pu être extraite.

In [12]:
df_openai_results = pd.read_parquet(OPENAI_BENCHMARK_PATH)
df_gemini_results = pd.read_parquet(GEMINI_BENCHMARK_PATH)
nbr_response = len(df_gemini_results)


### 8.1 Gestion des réponses non exploitables

Certaines réponses Gemini ne contiennent pas de lettre exploitable dans la variable `predicted_letter`. Ces observations sont identifiées puis retirées de la comparaison directe entre les deux modèles.

Le nombre de réponses ainsi exclues est conservé explicitement afin de ne pas confondre une absence de prédiction exploitable avec une réponse médicale incorrecte.

In [13]:
# on supprime les lignes où "predicted_letter" = "None"
df_gemini_results = df_gemini_results.dropna(
    subset=["predicted_letter"]
).reset_index(drop=True)

In [14]:
print("nombre de lignes sans réponse de gémini : ", nbr_response - len(df_gemini_results))

nombre de lignes sans réponse de gémini :  1226


### 8.2 Alignement des échantillons comparés

Pour garantir une comparaison équitable, seuls les `sample_id` disposant d'une réponse exploitable pour les deux modèles sont conservés.

L'intersection des identifiants Gemini et OpenAI définit ainsi le sous-ensemble utilisé pour la comparaison directe.

Cette étape garantit que les métriques comparatives sont calculées sur exactement les mêmes questions.

In [15]:
# on conserve uniquement les identité commune au 2 dataframe
common_sample_ids = set(
    df_gemini_results["sample_id"]
) & set(
    df_openai_results["sample_id"]
)

df_gemini_results = df_gemini_results[
    df_gemini_results["sample_id"].isin(common_sample_ids)
].reset_index(drop=True)

df_openai_results = df_openai_results[
    df_openai_results["sample_id"].isin(common_sample_ids)
].reset_index(drop=True)

In [16]:
print(f"nombre de réponse conservé de chat : ", len(df_openai_results))
print(f"nombre de réponse conservé de gémini : ", len(df_gemini_results))

nombre de réponse conservé de chat :  3791
nombre de réponse conservé de gémini :  3791


## 9. Comparaison question par question des modèles

Les prédictions Gemini et OpenAI sont regroupées afin de comparer leurs réponses sur chaque question commune.

Deux tableaux sont construits :

- `df_model_comparison_details`, qui rassemble les réponses et informations des deux modèles ;
- `df_model_disagreements`, qui isole les questions pour lesquelles leurs prédictions diffèrent.

Les désaccords sont particulièrement intéressants pour l'analyse qualitative, car ils permettent d'étudier les situations dans lesquelles les modèles interprètent différemment une même question médicale.

Un désaccord ne constitue cependant pas, à lui seul, une hallucination.

In [17]:
df_model_comparison_details, df_model_disagreements = build_model_comparison(df_gemini_results, df_openai_results, df_sample)
print("Réponses de gémini et chat dans un même Dataframe")
display(df_model_comparison_details)


Réponses de gémini et chat dans un même Dataframe


,sample_id,prompt_version,reference_letter,gemini_response,gemini_raw_response,gemini_is_correct,gemini_format_valid,gemini_justification,gemini_confidence,gemini_generation_error,openai_response,openai_raw_response,openai_is_correct,openai_format_valid,openai_justification,openai_confidence,openai_generation_error,question_context
0,mcqu_train_13934,prompt_v3,B,B,Réponse : B\nJustification : La sensibilité d'...,True,True,La sensibilité d'un test de dépistage correspo...,100.0,None,B,Réponse : B\nJustification : La sensibilité es...,True,True,La sensibilité est la capacité d’un test à êtr...,99,None,Question :\nDans un test de dépistage d'une ma...
1,mcqu_train_18036,prompt_v3,D,D,Réponse : D\nJustification : La patiente prése...,True,True,La patiente présente des métrorragies persista...,100.0,None,D,Réponse : D \nJustification : Il s’agit de mé...,True,True,Il s’agit de métrorragies persistantes chez un...,82,None,"Cas clinique :\nUne femme de 47 ans, nulligest..."
2,mcqu_train_9861,prompt_v3,B,B,Réponse : B\nJustification : L'instillation d'...,True,True,L'instillation d'un collyre mydriatique entraî...,100.0,None,B,Réponse : B \nJustification : Un collyre mydr...,True,True,Un collyre mydriatique peut déclencher ou aggr...,98,None,Question :\nLors de la prescription d'un colly...
3,mcqu_train_1065,prompt_v3,E,E,Réponse : E\nJustification : Toutes les affirm...,True,True,Toutes les affirmations sont exactes : 1) L'ap...,100.0,None,A,Réponse : A\nJustification : L’ischémie aiguë ...,False,True,L’ischémie aiguë est d’autant plus grave qu’el...,78,None,Question :\nParmi les affirmations suivantes l...
4,mcqu_train_27194,prompt_v3,B,B,Réponse : B\nJustification : Le tableau cliniq...,True,True,Le tableau clinique associe un syndrome de hyp...,95.0,None,B,Réponse : B \nJustification : Le tableau est ...,True,True,Le tableau est celui d’une hypothyroïdie franc...,92,None,"Cas clinique :\nPatiente âgée de 62 ans, sans ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3786,mcqu_train_588,prompt_v3,B,B,Réponse : B\nJustification : Un proto-oncogène...,True,True,Un proto-oncogène est un gène cellulaire norma...,100.0,None,B,Réponse : B \nJustification : Un proto-oncogè...,True,True,Un proto-oncogène est un gène normal présent d...,98,None,Question :\nUn proto-oncogène :\n\nProposition...
3787,mcqu_train_3475,prompt_v3,B,B,Réponse : B\nJustification : La réaction de cu...,True,True,La réaction de culture mixte lymphocytaire (CM...,100.0,None,B,Réponse : B\nJustification : La culture mixte ...,True,True,La culture mixte lymphocytaire met en évidence...,98,None,Question :\nQuelles sont les cellules qui répo...
3788,mcqu_train_12382,prompt_v3,D,D,Réponse : D\nJustification : L'intoxication ai...,True,True,L'intoxication aiguë par les morphiniques se c...,100.0,None,D,Réponse : D\nJustification : L’intoxication pa...,True,True,L’intoxication par les morphiniques provoque c...,98,None,Question :\nL'intoxication par les morphinique...
3789,mcqu_train_18662,prompt_v3,C,C,Réponse : C\nJustification : Le cromoglycate d...,True,True,Le cromoglycate de sodium (Lomudal°) est un an...,100.0,None,C,Réponse : C \nJustification : Le cromoglycate...,True,True,Le cromoglycate de sodium est un antiallergiqu...,98,None,Question :\nLe cromoglycate de sodium (Lomudal...


In [18]:
print("Comparaison des erreurs de réponse des 2 LLM")
display(df_model_disagreements)

Comparaison des erreurs de réponse des 2 LLM


,sample_id,prompt_version,question_context,reference_letter,gemini_response,gemini_is_correct,gemini_format_valid,gemini_justification,gemini_confidence,openai_response,openai_is_correct,openai_format_valid,openai_justification,openai_confidence
0,mcqu_train_1065,prompt_v3,Question :\nParmi les affirmations suivantes l...,E,E,True,True,Toutes les affirmations sont exactes : 1) L'ap...,100.0,A,False,True,L’ischémie aiguë est d’autant plus grave qu’el...,78
1,mcqu_train_14642,prompt_v3,Question :\nQuelle est la concentration en pro...,C,C,True,True,"Le lait maternel mature contient environ 1,0 à...",95.0,A,False,True,Le lait maternel nature contient en moyenne en...,98
2,mcqu_train_24386,prompt_v3,"Cas clinique :\nMr. R.B. demeurant à Alger, âg...",D,D,True,True,L'association de troubles du transit chronique...,100.0,A,False,True,Les douleurs abdominales chroniques avec diarr...,88
3,mcqu_train_21866,prompt_v3,"Cas clinique :\nUn homme âgé de 32 ans, origin...",D,D,True,True,La présence d'une fièvre avec frissons et d'un...,95.0,B,False,True,Le tableau évoque une infection biliaire compl...,82
4,mcqu_train_8311,prompt_v3,Cas clinique :\nUn patient de 45 ans est attei...,D,D,True,True,"Chez un homme de 45 ans, la valeur normale de ...",95.0,C,False,True,"Avant la binéphrectomie, l’anémie est essentie...",88
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
459,mcqu_train_25805,prompt_v3,"Cas clinique :\nMadame F.Z. 24 ans, à déjà acc...",C,C,True,True,La révision utérine n'est jamais systématique ...,95.0,B,False,True,"Après un accouchement prématuré, on réalise sy...",93
460,mcqu_train_6100,prompt_v3,Cas clinique :\nUn garçon de cinq ans en parfa...,B,C,False,True,Chez un enfant de 5 ans présentant une boiteri...,95.0,B,True,True,"Chez cet enfant de 5 ans, une boiterie aiguë a...",72
461,mcqu_train_24849,prompt_v3,"Question :\nEn cas de septicémie, l'antibiothé...",B,B,True,True,Le traitement antibiotique d'une septicémie (i...,95.0,D,False,True,"En cas de septicémie, l’antibiothérapie doit ê...",78
462,mcqu_train_21161,prompt_v3,Cas clinique :\nUn homme âgé de 65 ans consult...,B,D,False,True,L'occlusion de la veine centrale de la rétine ...,95.0,A,False,True,Une occlusion de la veine centrale de la rétin...,78


## 10. Identification des réponses incorrectes

Les réponses dont la lettre prédite diffère de la réponse de référence MediQAI sont extraites pour chacun des deux modèles.

Une même question peut apparaître deux fois dans ce tableau lorsque Gemini et OpenAI produisent tous deux une réponse incorrecte.

Cette représentation permet d'étudier séparément :

- le nombre total d'erreurs ;
- le nombre de questions distinctes concernées ;
- les questions échouées par les deux modèles ;
- les erreurs propres à chaque modèle.

In [19]:
df_model_errors = get_model_errors(
    df_gemini_results,
    df_openai_results,
    df_sample,
)

### 10.1 Répartition des erreurs selon les spécialités médicales

Les erreurs sont regroupées selon la variable `medical_subject` afin d'identifier les spécialités médicales dans lesquelles le plus grand nombre de réponses incorrectes est observé.

Pour chaque modèle, les trois spécialités comportant le plus grand nombre d'erreurs sont affichées.

Cette analyse est **descriptive** : un nombre élevé d'erreurs dans une spécialité ne signifie pas nécessairement que le modèle y est proportionnellement moins performant, car le nombre total de questions peut varier selon les spécialités. Une comparaison des taux d'erreur nécessiterait de rapporter le nombre d'erreurs au nombre de questions disponibles dans chaque spécialité.

In [20]:
number_distinct_ids = df_model_errors["sample_id"].nunique()
number_duplicate_ids = df_model_errors["sample_id"].duplicated().sum()
number_openai = (df_model_errors["model_name"] == "OpenAI").sum()
number_gemini = (df_model_errors["model_name"] == "Gemini").sum()


top_3_errors_by_model = (
    df_model_errors
    .groupby(
        ["model_name", "medical_subject"]
    )
    .size()
    .reset_index(name="number_errors")
    .sort_values(
        ["model_name", "number_errors"],
        ascending=[True, False],
    )
    .groupby("model_name")
    .head(3)
    .reset_index(drop=True)
)

### 10.2 Constitution du corpus d'erreurs

Les réponses incorrectes sont sauvegardées aux formats CSV et Parquet.

Ce corpus conserve les informations nécessaires à l'analyse qualitative des réponses, notamment le contexte de la question, la réponse de référence, la prédiction du modèle, sa justification et son niveau de confiance lorsqu'ils sont disponibles.

Ces erreurs constituent des **candidats à l'analyse des hallucinations**. Une réponse incorrecte n'est pas automatiquement considérée comme une hallucination : le contenu de sa justification doit être examiné afin de déterminer la nature exacte de l'erreur.

In [21]:
print(len(df_model_errors), " réponses incorrectes")
print(f"\n")
print(number_distinct_ids, " questions distinctes")
print(number_duplicate_ids, " questions ratées par les deux modèles")
print(f"\n")
print(number_gemini, " erreurs de Gemini")
print(number_openai, " erreurs d'OpenAI")
display(df_model_errors)


813  réponses incorrectes


616  questions distinctes
197  questions ratées par les deux modèles


284  erreurs de Gemini
529  erreurs d'OpenAI


,model_name,sample_id,medical_subject,question_context,reference_letter,model_response,is_correct,justification,confidence
0,Gemini,mcqu_train_23245,Orthopedics,"Cas clinique :\nUn jeune homme de 25 ans, vict...",E,B,False,L'attitude vicieuse du membre inférieur droit ...,95.0
1,Gemini,mcqu_train_11747,Infectious Diseases,Cas clinique :\nVous êtes amené à voir à votre...,B,A,False,Devant une angine streptococcique (à streptoco...,95.0
2,Gemini,mcqu_train_7689,Nephro-Urology,Question :\nUne insuffisance rénale aiguë par ...,A,B,False,L'insuffisance rénale aiguë par obstacle est p...,100.0
3,Gemini,mcqu_train_18793,Pulmonology,Cas clinique :\nUn homme de 40 ans a ressenti ...,B,C,False,Chez un patient de 40 ans (ou de plus de 40 an...,90.0
4,Gemini,mcqu_train_12851,Pulmonology,"Cas clinique :\nMonsieur M., manoeuvre de son ...",C,A,False,L'image décrit une opacité en bande d'environ ...,90.0
...,...,...,...,...,...,...,...,...,...
808,OpenAI,mcqu_train_948,Cardiology,Question :\nQuelle est la dose de xylocaïne à ...,C,B,False,La xylocaïne (lidocaïne) utilisée en préventio...,88.0
809,OpenAI,mcqu_train_25805,Gynecology and Obstetrics,"Cas clinique :\nMadame F.Z. 24 ans, à déjà acc...",C,B,False,"Après un accouchement prématuré, on réalise sy...",93.0
810,OpenAI,mcqu_train_24849,Microbiology,"Question :\nEn cas de septicémie, l'antibiothé...",B,D,False,"En cas de septicémie, l’antibiothérapie doit ê...",78.0
811,OpenAI,mcqu_train_21161,Ophthalmology,Cas clinique :\nUn homme âgé de 65 ans consult...,B,A,False,Une occlusion de la veine centrale de la rétin...,78.0


### 10.3 Analyse de la confiance déclarée

Le `prompt_v3` demande à chaque modèle de fournir, en plus de sa réponse et de sa justification, un **niveau de confiance déclaré**.

Cette information permet d'étudier la relation entre la confiance exprimée par le modèle et l'exactitude de sa réponse. Une attention particulière est portée aux **réponses incorrectes associées à une confiance élevée**.

Ces situations sont particulièrement intéressantes dans le cadre de l'étude des hallucinations : un modèle peut produire une réponse erronée tout en présentant son raisonnement avec un niveau de confiance important.

Cependant, une confiance élevée associée à une réponse incorrecte ne constitue pas, à elle seule, une preuve d'hallucination. Ces observations sont considérées comme des **cas prioritaires pour l'analyse qualitative** de la justification produite par le modèle.

In [25]:
# Vérification des valeurs de confiance
for model_name, df in [
    ("Gemini", df_gemini_results),
    ("OpenAI", df_openai_results),
]:
    print(f"\n{model_name}")
    print(df["declared_confidence"].describe())
    print("Valeurs manquantes :", df["declared_confidence"].isna().sum())


Gemini
count    3790.000000
mean       98.343536
std         2.626917
min        80.000000
25%        95.000000
50%       100.000000
75%       100.000000
max       100.000000
Name: declared_confidence, dtype: float64
Valeurs manquantes : 1

OpenAI
count    3791.000000
mean       93.693748
std         6.731712
min        33.000000
25%        92.000000
50%        96.000000
75%        98.000000
max       100.000000
Name: declared_confidence, dtype: float64
Valeurs manquantes : 0


#### Confiance selon l'exactitude de la réponse

La confiance moyenne est comparée entre les réponses correctes et incorrectes.

Cette analyse permet d'observer si les modèles tendent à déclarer une confiance plus importante lorsqu'ils produisent une réponse correcte et, inversement, s'ils sont capables d'exprimer davantage d'incertitude lorsqu'ils se trompent.

In [27]:
def confidence_by_correctness(df, model_name):
    df_conf = df.dropna(subset=["declared_confidence"]).copy()

    summary = (
        df_conf
        .groupby("is_correct")["declared_confidence"]
        .agg(["count", "mean", "median", "std"])
        .reset_index()
    )

    summary["model"] = model_name

    return summary


gemini_confidence_summary = confidence_by_correctness(
    df_gemini_results,
    "Gemini"
)

openai_confidence_summary = confidence_by_correctness(
    df_openai_results,
    "OpenAI"
)

df_confidence_summary = pd.concat(
    [gemini_confidence_summary, openai_confidence_summary],
    ignore_index=True
)

df_confidence_summary

,is_correct,count,mean,median,std,model
0,False,283,95.332155,95.0,3.190621,Gemini
1,True,3507,98.586541,100.0,2.418228,Gemini
2,False,529,86.748582,88.0,10.251160,OpenAI
3,True,3262,94.820049,96.0,5.153497,OpenAI


#### Identification des erreurs à confiance élevée

Les réponses incorrectes associées à une confiance déclarée supérieure ou égale à **90 %** sont isolées.

Le seuil de 90 % est utilisé ici comme critère opérationnel pour identifier les situations dans lesquelles le modèle exprime une forte certitude malgré une réponse incompatible avec la référence.

Ces observations seront examinées en priorité lors de l'analyse qualitative des hallucinations.

In [38]:
HIGH_CONFIDENCE_THRESHOLD = 90

number_error = [number_gemini, number_openai]

# Sélection des erreurs avec une confiance élevée
df_high_confidence_errors = df_model_errors[
    (df_model_errors["confidence"].notna()) &
    (df_model_errors["confidence"] >= HIGH_CONFIDENCE_THRESHOLD)
].copy()

# Affichage pour chaque modèle
for model, numerror in zip(["Gemini", "OpenAI"], number_error):

    n_errors = len(
        df_high_confidence_errors[
            df_high_confidence_errors["model_name"] == model
        ]
    )

    print(
        f"Nombre d'erreurs avec une confiance pour {model} "
        f">= {HIGH_CONFIDENCE_THRESHOLD}% : {n_errors}"
    )

    print(
        f"Taux d'erreurs à forte confiance pour {model} : "
        f"{n_errors / numerror:.2%}"
    )

    print()

Nombre d'erreurs avec une confiance pour Gemini >= 90% : 281
Taux d'erreurs à forte confiance pour Gemini : 98.94%

Nombre d'erreurs avec une confiance pour OpenAI >= 90% : 252
Taux d'erreurs à forte confiance pour OpenAI : 47.64%



#### Interprétation des erreurs à forte confiance

L'analyse met en évidence une différence importante entre les deux modèles concernant la confiance déclarée lors des réponses incorrectes.

Pour **Gemini**, 281 erreurs présentent une confiance supérieure ou égale à 90 %, soit **98,94 % de l'ensemble de ses réponses incorrectes**. La quasi-totalité des erreurs produites par Gemini sont donc associées à un niveau de confiance très élevé.

Pour **OpenAI**, 252 erreurs présentent une confiance supérieure ou égale à 90 %, soit **47,64 % de ses réponses incorrectes**. Les erreurs à forte confiance restent donc fréquentes, mais leur proportion est nettement inférieure à celle observée pour Gemini.

Cette différence suggère que, dans les conditions de cette expérimentation, la confiance déclarée par Gemini discrimine peu les réponses correctes des réponses incorrectes. OpenAI semble davantage moduler son niveau de confiance lorsqu'il produit une réponse erronée.

Ces résultats doivent néanmoins être interprétés avec prudence. La confiance déclarée correspond à une information générée par le modèle lui-même et ne constitue pas une probabilité calibrée d'exactitude. Par ailleurs, une réponse incorrecte associée à une confiance élevée ne constitue pas automatiquement une hallucination.

Les erreurs à forte confiance sont donc considérées comme des **cas prioritaires pour l'analyse qualitative des justifications**, et non comme des hallucinations déjà identifiées.

In [22]:
save_csv(df_model_errors, RESULTS_DIR, "erreur_LLM.csv")
save_parquet(df_model_errors, RESULTS_DIR, "erreur_LLM.parquet")

WindowsPath('c:/Users/MANEL/Dropbox/projet_evaluation_LLM_medicale/final_evaluation/erreur_LLM.parquet')

## 11. Calcul des métriques de performance

Deux métriques principales sont calculées pour comparer les prédictions des modèles aux réponses de référence.

### Exactitude (*accuracy*)

L'exactitude correspond à la proportion de questions pour lesquelles la lettre prédite est identique à la lettre de référence :


**Accuracy = correct/N_total**


Cette métrique fournit une mesure globale et directement interprétable des performances sur les QCM.

### Score F1 macro

Un score F1 est également calculé séparément pour chacune des cinq classes de réponse (`A`, `B`, `C`, `D`, `E`), puis moyenné sans pondération :


**F1_macro = 1/K * sum{k=1}^{K}F1_k**


avec **K = 5**.

Le F1 macro accorde le même poids à chaque lettre de réponse, indépendamment de sa fréquence dans le dataset. Il complète ainsi l'exactitude lorsque la distribution des réponses correctes entre les différentes lettres n'est pas parfaitement équilibrée.

In [23]:
from sklearn.metrics import f1_score

def calculate_accuracy_metrics(df, string):
    valid_results = df["is_correct"].dropna()
    name = string,
    total_experiments = len(df)
    valid_experiments = len(valid_results)
    correct_answers = valid_results.sum()
    incorrect_answers = valid_experiments - correct_answers

    accuracy = valid_results.mean()

    return pd.DataFrame(
        [{
            "model":name,
            "total_experiments": total_experiments,
            "valid_experiments": valid_experiments,
            "correct_answers": int(correct_answers),
            "incorrect_answers": int(incorrect_answers),
            "accuracy": accuracy,
        }]
    )


def calculate_f1_score(df):
    valid_results = df.dropna(
        subset=[
            "reference_letter",
            "predicted_letter",
        ]
    )

    score = f1_score(
        valid_results["reference_letter"],
        valid_results["predicted_letter"],
        labels=["A", "B", "C", "D", "E"],
        average="macro",
        zero_division=0,
    )

    return score

### 11.1 Performance globale des modèles

Les fonctions précédentes sont appliquées séparément aux résultats Gemini et OpenAI.

Pour chaque modèle sont calculés :

- le nombre total d'expériences ;
- le nombre d'expériences disposant d'un résultat exploitable ;
- le nombre de réponses correctes ;
- le nombre de réponses incorrectes ;
- l'exactitude ;
- le score F1 macro.

Ces résultats constituent les principales métriques quantitatives de comparaison des deux modèles sur l'échantillon évalué.

In [24]:
precision_gemini = calculate_accuracy_metrics(df_gemini_results, "Gemini")
precision_openai = calculate_accuracy_metrics(df_openai_results, "OpenAI")


f1_gemini = calculate_f1_score(df_gemini_results)
f1_openai = calculate_f1_score(df_openai_results)


print(f"Score F1 gemini : {f1_gemini:.3f}")
print(f"Score F1 openai : {f1_openai:.3f}")
display(precision_gemini)
display(precision_openai)

Score F1 gemini : 0.921
Score F1 openai : 0.855


,model,total_experiments,valid_experiments,correct_answers,incorrect_answers,accuracy
0,"(Gemini,)",3791,3791,3507,284,0.925086


,model,total_experiments,valid_experiments,correct_answers,incorrect_answers,accuracy
0,"(OpenAI,)",3791,3791,3262,529,0.860459


## Conclusion

Ce notebook a permis d'appliquer à grande échelle le protocole expérimental développé dans les étapes précédentes afin de comparer Gemini et OpenAI sur un même ensemble de questions médicales MCQU.

Après contrôle des réponses exploitables, les prédictions des deux modèles ont été alignées afin de garantir une comparaison question par question sur un ensemble commun. L'évaluation quantitative repose principalement sur l'exactitude et le score F1 macro, complétés par une analyse des erreurs, des désaccords et de la confiance déclarée par les modèles.

L'analyse de la confiance met notamment en évidence une différence importante entre les deux modèles. Parmi leurs réponses incorrectes, **98,94 % des erreurs de Gemini** présentent une confiance supérieure ou égale à 90 %, contre **47,64 % pour OpenAI**. Ces résultats montrent l'intérêt d'étudier non seulement l'exactitude des réponses, mais également le degré de certitude exprimé par les modèles lorsqu'ils se trompent.

Les réponses incorrectes constituent désormais un corpus de cas candidats pour l'analyse qualitative. Elles ne sont toutefois pas automatiquement considérées comme des hallucinations : l'étude de leurs justifications reste nécessaire pour déterminer si elles contiennent des affirmations médicales incorrectes, non étayées, contradictoires ou inventées.

Ce corpus constitue ainsi le point de départ de l'étape suivante du projet, consacrée à **l'annotation et à la détection des hallucinations dans les réponses des LLM**.